# GX_07
# Transmission Matrices for Atmosphere Correction

In this exercise, you are use a transmission matrix to correct for the effects of atmospheric turbulence on astronomical measurements.

A common approach for image correction in astronomy is to use a laser guide star. The guide star is an artificial point of light in the sky created by using a laser beam to energize atoms in the mesosphere which re-emit the laser light in turn. These guide stars serve as a known object to image through the atmosphere, and after measuring the phase and amplitude of the light from the guide star back on earth, the signal can be used to reverse engineer the properties of the atmosphere.

Because the atmosphere may be rapidly changing, this calculation is often done in real time so that the performance of the imaging (or other applications such as satellite communication) is always maximized. This category of real-time correction is called adaptive optics.

For this exercise, you will practice turning the measurements from a guidestar that is swept across the sky into a transmission matrix and then demonstrate how this can be used to correct an image of a galaxy taken through the atmosphere.


In [ ]:
# - No modification necessary -

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, transform

# -------------------------
# Physical parameters
# -------------------------
wavelength = 1e-6
k0 = 2 * np.pi / wavelength

# -------------------------
# Grid parameters
# -------------------------
N = 128
dx = 5e-2
L = N * dx

x = np.linspace(-L/2, L/2, N)
y = np.linspace(-L/2, L/2, N)
X, Y = np.meshgrid(x, y)

# -------------------------
# Frequency coordinates
# -------------------------
kx = np.fft.fftfreq(N, d=dx) * 2 * np.pi
ky = np.fft.fftfreq(N, d=dx) * 2 * np.pi
KX, KY = np.meshgrid(kx, ky)
K2 = KX**2 + KY**2

# -------------------------
# Propagation parameters
# -------------------------
dz = 5e3          # meters between phase screens
num_screens = 18     # number of atmospheric layers

# Total propagation distance ~ 90 km
total_distance = dz * num_screens

def propagate_free_space(field, distance):
    """
    Angular spectrum propagation.
    """
    
    kz = np.sqrt((k0**2 - K2.astype(complex)))
    H = np.exp(1j * kz * distance)
    
    return np.fft.ifft2(np.fft.fft2(field) * H)

# -------------------------
# Plotting Helpers
# -------------------------
def plot_intensity(field, title="Field"):
    plt.imshow(np.abs(field)**2, extent=[x.min(), x.max(), y.min(), y.max()])
    plt.title(title + " - Intensity")
    plt.colorbar()
    plt.show()
    
def plot_phase(field, title="Field"):
    plt.imshow(np.angle(field), extent=[x.min(), x.max(), y.min(), y.max()], cmap='twilight', vmin=0, vmax=2*np.pi)
    plt.title(title + " - Phase")
    plt.colorbar()
    plt.show()

def plot_field(field, title="Field"):
    intensity = np.abs(field)**2
    phase = np.angle(field)

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))

    im0 = axs[0].imshow(intensity, extent=[x.min(), x.max(), y.min(), y.max()])
    axs[0].set_title(title + " - Intensity")
    plt.colorbar(im0, ax=axs[0])

    im1 = axs[1].imshow(phase, extent=[x.min(), x.max(), y.min(), y.max()], cmap='twilight', vmin=0, vmax=2*np.pi)
    axs[1].set_title(title + " - Phase")
    plt.colorbar(im1, ax=axs[1])

    plt.tight_layout()
    plt.show()

# Part 1

In the first part of this exercise, you will build up the propagation algorithm to travel through the atmosphere and the inputs to propagate. To keep complexity relatively low, you will use a simple approximation of the atmosphere where turbulence manifests as small phase changes that are accumulated over kilometers. 

The required code to generate the atmospheric phase masks using Von Karmen statistics has been provided for you (and is sourced from the aotools library). You should take some time to look at how the algorithm works, but you will not be required to handle any of it yourself.

Start by implementing the function generate_guide_star_basis. This function should create a uniform grid of Gaussian spots over the image. The parameter num_points_per_axis encodes how many spots should be used in each row (and how many rows should be used), and the spot size encodes the beam radius of each Gaussian spot. Extent represents the amount of the image that the sampling grid should cover, and is slightly less than 1 so that the edges are not included as valid spot sites.

To propagate through the atmosphere, you will use beam propagation with these atmospheric phase masks for phase correction. Implement this in the function propagate_through_atmosphere, then implement propagate_all_guide_stars to calculate the outputs for all inputs.

After you have completed this, run the provided script to generate the inputs and their corresponding outputs and plot some of the results. Then, answer the following questions:
1) Why are we able to propagate over kilometers here, when normally in BPM we only go a few centimeters?
2) Why can we make the thin phase mask approximation in this scenario when our "thin" masks are really kilometers thick? 

In [ ]:
# - No modification necessary
# - Sourced from the aotools library with minor changes to random seeding behavior

def ift2(G, delta_f, FFT=None):
    """
    Wrapper for inverse fourier transform

    Parameters:
        G: data to transform
        delta_f: pixel seperation
        FFT (FFT object, optional): An accelerated FFT object
    """

    N = G.shape[0]

    if FFT:
        g = np.fft.fftshift(FFT(np.fft.fftshift(G))) * (N * delta_f) ** 2
    else:
        g = np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(G))) * (N * delta_f) ** 2

    return g

def ft_phase_screen(r0, N, delta, L0, l0, FFT=None):
    '''
    Creates a random phase screen with Von Karmen statistics.
    (Schmidt 2010)
    
    Parameters:
        r0 (float): r0 parameter of scrn in metres
        N (int): Size of phase scrn in pxls
        delta (float): size in Metres of each pxl
        L0 (float): Size of outer-scale in metres
        l0 (float): inner scale in metres

    Returns:
        ndarray: numpy array representing phase screen
    '''
    delta = float(delta)
    r0 = float(r0)
    L0 = float(L0)
    l0 = float(l0)

    del_f = 1./(N*delta)

    fx = np.arange(-N/2., N/2.) * del_f

    (fx, fy) = np.meshgrid(fx,fx)
    f = np.sqrt(fx**2. + fy**2.)

    fm = 5.92/l0/(2*np.pi)
    f0 = 1./L0

    PSD_phi = (0.023*r0**(-5./3.) * np.exp(-1*((f/fm)**2)) / (((f**2) + (f0**2))**(11./6)))

    PSD_phi[int(N/2), int(N/2)] = 0

    cn = ((np.random.normal(size=(N, N))+1j * np.random.normal(size=(N, N))) * np.sqrt(PSD_phi)*del_f)

    phs = ift2(cn, 1, FFT).real

    return phs

def generate_atmosphere(num_screens, r0, l0, L0, seed=0):
    """
    Generate a list of phase screens representing atmosphere.
    """
    np.random.seed(seed)
    
    screens = []
    for _ in range(num_screens):
        phase_screen = ft_phase_screen(r0, N, dx, L0, l0)
        screen = np.exp(1j * phase_screen)
        screens.append(screen)
    
    return screens

# -------------------------
# Generate atmosphere
# -------------------------
r0 = 2
l0 = 0.5
L0 = 100
phase_screens = generate_atmosphere(num_screens, r0, l0, L0)

for i in range(min(2, len(phase_screens))):
    plot_phase(phase_screens[i], f"Screen {i+1}")

In [ ]:
def generate_guide_star_basis(num_points_per_axis, spot_size, extent=0.9):
    """
    Generate guide stars across object plane.
    """
    
    raise NotImplementedError

In [ ]:
def propagate_through_atmosphere(field, phase_screens):
    """
    Propagate field through multiple phase screens.
    """
    
    raise NotImplementedError

def propagate_all_guide_stars(input_basis, phase_screens):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

# ---------------------------
# Generate Inputs and Outputs
# ---------------------------
num_points = 64   # reduce for speed
spot_size = 1e-1

input_basis, _ = generate_guide_star_basis(num_points, spot_size, extent=0.64)
output_basis = propagate_all_guide_stars(input_basis, phase_screens)

for i in range(min(2, len(input_basis))):
    plot_field(input_basis[i], f"Input {i+1}")
    plot_field(output_basis[i], f"Output {i+1}")     

## Discussion
TODO

# Part 2

Now that you have recorded the output of atmospheric propagation for each guide star position, you must create the transmission matrix from the data. Implement this in the below function.


In [ ]:
def build_transmission_matrix(inputs, outputs):
    raise NotImplementedError

In [ ]:
# - No modification necessary -

TM = build_transmission_matrix(input_basis, output_basis)

# Part 3

Now it is time to test your transmission matrix. You have been provided with an image and what it looks like when it arrives at your detector after propagating through the atmosphere. Complete the reconstruct_input and reconstruct_field_from_basis functions to retrieve the original image.

Then, answer the following questions:
1) Why does our reconstruction only cover part of the image field?
2) What could we change to increase the resolution of our corrected image without directly increasing the resolution of our detector?

In [ ]:
# - No modification necessary -

def load_test_image():
    """
    Load, crop to square, and resize grayscale test image.
    """
    # Convert to grayscale
    image = color.rgb2gray(data.hubble_deep_field())

    # Get original dimensions
    h, w = image.shape

    # Determine square crop size (cN)
    c = 0.2  # for example
    crop_size = int(min(h, w) * c)

    # Compute top-left corner for centered crop
    start_h = (h - crop_size) // 2
    start_w = (w - crop_size) // 2

    # Crop to square (cN x cN)
    image_cropped = image[start_h:start_h + crop_size,
                          start_w:start_w + crop_size]

    # Resize to (N x N)
    image_resized = transform.resize(image_cropped, (N, N))

    return image_resized

# -------------------------
# Load object
# -------------------------
image = load_test_image()

def propagate_image_through_atmosphere(image, phase_screens):
    """
    Treat image as field amplitude and propagate.
    """
    
    field = image.astype(complex)
    
    output = propagate_through_atmosphere(field, phase_screens)
    
    return output

# -------------------------
# Forward propagation
# -------------------------
distorted = propagate_image_through_atmosphere(image, phase_screens)
plot_field(distorted, "Distorted Image")

In [ ]:
def reconstruct_input(TM, output_field):
    """
    Recover input via least squares.
    """
    
    raise NotImplementedError

def reconstruct_field_from_basis(coeffs, basis):
    
    raise NotImplementedError

In [ ]:
# - No modification necessary -

coeffs = reconstruct_input(TM, distorted)
recovered = reconstruct_field_from_basis(coeffs, input_basis)

plt.imshow(np.abs(recovered), cmap='gray_r')
plt.title("Reconstructed Image")
plt.show()

## Discussion
TODO

# Bonus

In this exercise, we made the strong assumption that we could create a sodium beacon-based laser guide star that would act as a perfect known source and could be swept about the sky to build a transmission matrix. Think about how a sodium beacon laser guide star system would work (or read about it!) and reflect on at least 3 different complications that would arise in a real system like this.

## Discussion
TODO